# Collection trajectories on the cost heatmaps

This viewer reads the completed wider one-macro run from Drive. It plots Bands 1–4 for each of the six collection goals. Band 1 uses raw MSE. Bands 2–4 use the checkpoint from the preceding band, which is what the planner used for collection.

Green lines show every recorded simulator step of a successful rollout. Red dashed lines connect the start and final positions of failed rollouts. Circles mark starts and crosses mark ends. Small white dots mark the recorded observations retained as endpoints of adapter training pairs. Amber squares outline the start band. Each panel has its own unclipped color scale and spans the right room through x = 60.

Use a GPU for all six goals. Set `GOAL_INDICES = [0]` for a CPU preview of the saved heatmap for (50, 42). Other maps require encoding the position grid once with the frozen encoder. The notebook runs no planning or training and saves figures to the local Colab runtime.

In [ ]:
from collections import defaultdict
from pathlib import Path
import json

from google.colab import drive
import numpy as np
import torch

drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive')
RUN = (DRIVE / 'datasets/curriculum_cost_adapter/experience_one_macro_adapter_wide_v1'
       / '20261108T091827483705Z')
PLOT_DIR = Path('/content/experience_band_trajectories')
GOAL_INDICES = None  # None selects all collection goals.

def read_json(path):
    return json.loads(path.read_text())

config = read_json(RUN / 'experiment.json')
manifest = config['manifest']
report = read_json(RUN / 'comparison.json')
grid = report['grid']
grid_ids = grid['point_ids']
grid_shape = (len(grid['ys']), len(grid['xs']))
band_numbers = range(1, 5)
goals = {case['goal_index']: tuple(case['goal']) for case in manifest['stages'][0]}
selected = sorted(goals) if GOAL_INDICES is None else list(GOAL_INDICES)
if not selected or any(index not in goals for index in selected):
    raise ValueError(f'Choose collection goal indices from {sorted(goals)}.')

maps = {}
report_goals = {tuple(row['goal']): i for i, row in enumerate(report['methods']['raw'])}
for index in selected:
    if goals[index] in report_goals:
        report_index = report_goals[goals[index]]
        for band in band_numbers:
            method = 'raw' if band == 1 else f'band_{band - 1}'
            costs = report['methods'][method][report_index]['adapted_costs']
            maps[band, index] = np.asarray(costs)[grid_ids].reshape(grid_shape)
missing = [index for index in selected if (1, index) not in maps]

## Read the recorded paths

Band 1 was imported from the archived raw-MSE collection. The viewer follows that archive path recorded in the run configuration. For later bands it reads this run's own episode records.

In [ ]:
def load_paths(run, config, report, selected, band_numbers):
    paths = defaultdict(list)
    for band in band_numbers:
        name = f'band_{band}'
        stage = report['checkpoints'][name]
        archive = config.get('band1_source') if band == 1 else None
        root = Path(archive['root']) if archive else run
        runtime_id = archive['runtime_id'] if archive else stage['runtime_id']
        for row in stage['collection']:
            case = row['case']
            index = case['goal_index']
            if index not in selected:
                continue
            case_id = case['case_id']
            record = read_json(root / 'episodes' / name / f'{case_id}.json')
            if (record['case'] != case or record['runtime_id'] != runtime_id
                    or record['success'] != row['success']):
                raise ValueError(f'Episode provenance differs for {case_id}')
            if record['success']:
                raw = torch.load(root / record['raw_path'], map_location='cpu',
                                 weights_only=True, mmap=True)
                if raw['case'] != case or raw['runtime_id'] != runtime_id or not raw['success']:
                    raise ValueError(f'Raw trajectory provenance differs for {case_id}')
                xy = raw['states'][:, :2].numpy()
                if len(xy) != record['micro_steps'] + 1:
                    raise ValueError(f'Incomplete simulator-step trajectory for {case_id}')
            else:
                positions = record['achieved_positions']
                xy = np.asarray([positions[0], positions[-1]])
            retained = np.asarray(row['retained_positions']).reshape(-1, 2)
            paths[band, index].append((record['success'], xy, retained))
    return paths

paths = load_paths(RUN, config, report, selected, band_numbers)
print(f'Loaded {sum(map(len, paths.values()))} rollouts for {len(selected)} goal(s).')

## Fill in any missing goal heatmaps

This cell skips the encoder when all selected goals already have saved heatmaps. For other goals, it scores real rendered grid observations against the saved goal tokens, with the checkpoint that was active during collection.

In [ ]:
if missing:
    import shutil
    import subprocess
    import sys
    from types import SimpleNamespace

    if not torch.cuda.is_available():
        raise RuntimeError('Select a GPU runtime for all goals, or set GOAL_INDICES = [0] for a CPU preview.')

    repo = Path('/content/experience-trajectory-viewer-src')
    renderer = Path('/content/experience-trajectory-viewer-dino')
    for folder, url, commit in (
        (repo, 'https://github.com/kmylonas/i-jepa-wm.git', '9efadffa2c4364fe8e656a7b421eaa9fb6f87a06'),
        (renderer, 'https://github.com/gaoyuezhou/dino_wm.git', '0a9492fa12044b852ae9e001cc74604b79c8bb0c'),
    ):
        if not folder.exists():
            subprocess.run(['git', 'clone', url, str(folder)], check=True)
        subprocess.run(['git', '-C', str(folder), 'checkout', '--detach', commit], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gym==0.26.2',
                    'omegaconf', 'pyyaml', 'scipy', 'einops', 'imageio', 'imageio-ffmpeg'], check=True)
    sys.path.insert(0, str(repo))

    from src.mylonas_ijepa_wm.curriculum_replay import load_adapter
    from src.mylonas_ijepa_wm.curriculum_runtime import PilotRuntime, file_sha256, visual
    from src.mylonas_ijepa_wm.environment_adapters import WallAdapter
    from src.mylonas_ijepa_wm.environment_configs import EnvironmentConfig
    from src.mylonas_ijepa_wm.experience_curriculum import _encode_positions
    from src.mylonas_ijepa_wm.radial_multigoal_experiment import _scores

    signature = read_json(RUN / 'runtime.json')['signature']
    design = read_json(Path(config['handcrafted']['root']) / 'data' / 'design.json')
    if design['grid'] != grid:
        raise ValueError('Saved grid differs from the frozen dataset.')
    for index in selected:
        if tuple(design['points'][manifest['goal_ids'][index]]) != goals[index]:
            raise ValueError(f'Frozen goal position differs for collection goal {index}.')

    encoder_path = Path('/content/experience-trajectory-viewer-encoder.pth.tar')
    expected = signature['models']['encoder']
    if not encoder_path.exists() or file_sha256(encoder_path) != expected:
        shutil.copy2(DRIVE / 'checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar', encoder_path)
    if file_sha256(encoder_path) != expected:
        raise ValueError('Frozen encoder checksum differs from this run.')

    goal_path = RUN / 'collection_goals.pt'
    stages = report['checkpoints']
    if file_sha256(goal_path) != stages['band_1']['goal_sha256']:
        raise ValueError('Saved collection goal tokens differ.')
    targets = torch.load(goal_path, map_location='cpu', weights_only=True)
    if targets['runtime_id'] != stages['band_1']['runtime_id']:
        raise ValueError('Collection goal tokens belong to a different run.')

    device = torch.device('cuda')
    encoder = visual.load_frozen_ijepa_encoder(encoder_path, device)
    env = WallAdapter(EnvironmentConfig(env='wall', wall_layouts=[[32, 30]],
                                        dino_wm_root=str(renderer)), 100)
    runtime = SimpleNamespace(env=env, device=device, precision=signature['precision'],
                              args=SimpleNamespace(encoder_batch_size=signature['encoder_batch_size']),
                              load_encoder=lambda: encoder)
    runtime.encode_frames = lambda frames: PilotRuntime.encode_frames(runtime, frames)
    try:
        print(f'Encoding {len(grid_ids)} rendered grid positions once.')
        tokens = _encode_positions(runtime, design, grid_ids)
    finally:
        env.close()
        del runtime, encoder

    ref_ids = list(range(len(grid_ids), len(grid_ids) + len(selected)))
    tokens = torch.cat((tokens, targets['tokens'][selected]))
    for band in band_numbers:
        adapter = None
        if band > 1:
            prior = f'band_{band - 1}'
            checkpoint = RUN / prior / 'adapter.pt'
            expected_checkpoint = stages[prior]['checkpoint_sha256']
            if (stages[f'band_{band}']['collector'] != expected_checkpoint
                    or file_sha256(checkpoint) != expected_checkpoint):
                raise ValueError(f'Collector checkpoint differs for Band {band}.')
            adapter, _ = load_adapter(checkpoint, device, expected_runtime_id=stages[prior]['runtime_id'])
        scores = _scores(adapter, tokens, ref_ids, device)
        for index, row in zip(selected, scores):
            maps[band, index] = row[:len(grid_ids)].reshape(grid_shape)

## Plot Bands 1–4

The success count in each title refers to that band's collection cases for the plotted goal.

In [ ]:
from IPython.display import Image, display
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
import matplotlib.patheffects as effects
import matplotlib.pyplot as plt

def plot_goal(index, goal, bands, maps, paths, grid, plot_dir):
    gx, gy = goal
    fig, axes = plt.subplots(1, 4, figsize=(14, 6.6), layout='constrained')
    success_color, fail_color, band_color = '#62ef8c', '#ff7171', '#ffb33d'
    for band, ax in enumerate(axes, 1):
        costs = maps[band, index]
        mesh = ax.pcolormesh(grid['xs'], grid['ys'], costs, shading='nearest',
                             cmap='viridis', vmin=0, vmax=max(float(costs.max()), 1e-9))
        inner, outer = bands[band - 1]
        for radius in (inner, outer):
            if radius:
                ax.add_patch(Rectangle((gx-radius, gy-radius), 2*radius, 2*radius, fill=False,
                                       edgecolor=band_color, linewidth=1.2, linestyle=':',
                                       path_effects=[effects.withStroke(linewidth=2.4, foreground='#26262c')]))
        episodes = paths[band, index]
        for success, xy, retained in sorted(episodes, key=lambda item: item[0]):
            color = success_color if success else fail_color
            ax.plot(*xy.T, color=color, linestyle='-' if success else '--', linewidth=1.45,
                    zorder=4, path_effects=[effects.withStroke(linewidth=2.6, foreground='#26262c')])
            if len(retained):
                ax.scatter(*retained.T, s=12, facecolors='white', edgecolors='#26262c',
                           linewidths=.45, zorder=5)
            ax.scatter(*xy[0], s=32, facecolors='none', edgecolors=color, zorder=6)
            ax.scatter(*xy[-1], s=34, color=color, marker='x', zorder=6)
        ax.scatter(*goal, marker='*', s=115, color='black', edgecolor='white', zorder=6)
        successes = sum(success for success, _, _ in episodes)
        collector = 'Raw MSE' if band == 1 else f'After Band {band - 1} fit'
        ax.set(title=f'Band {band} · {inner}–{outer}\n{collector} · {successes}/{len(episodes)} success',
               xlabel='Ball x', xlim=(35, 60), ylim=(4, 60), aspect='equal')
        fig.colorbar(mesh, ax=ax, shrink=.62, label='Latent MSE')
    axes[0].set_ylabel('Ball y')
    fig.suptitle(f'Collection goal {index} ({gx:.2f}, {gy:.2f})')
    fig.legend(handles=[
        Line2D([], [], color='#188342', label='Successful path'),
        Line2D([], [], color='#cb363d', linestyle='--', label='Failed start to end'),
        Line2D([], [], marker='o', markerfacecolor='white', markeredgecolor='#26262c',
               markersize=4, linestyle='none', label='Training observation'),
        Line2D([], [], color='#555', marker='o', markerfacecolor='none', linestyle='none', label='Start'),
        Line2D([], [], color='#555', marker='x', linestyle='none', label='End'),
        Line2D([], [], color='black', marker='*', linestyle='none', label='Goal'),
        Line2D([], [], color='#b36300', linestyle=':', label='Start band edges'),
    ], loc='upper center', bbox_to_anchor=(.5, -.005), ncol=7, frameon=False)
    path = plot_dir / f'collection_goal_{index}_bands_1_to_4.png'
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return path

PLOT_DIR.mkdir(parents=True, exist_ok=True)
for index in selected:
    path = plot_goal(index, goals[index], manifest['bands'], maps, paths, grid, PLOT_DIR)
    display(Image(filename=str(path)))